In [ ]:
!pip install -r requirements.txt --q

In [ ]:
import datasets
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForQuestionAnswering, Trainer, TrainingArguments
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer as SummarizationTokenizer
import evaluate
import os
import json
import torch
import torch.multiprocessing as mp

mp.set_start_method('spawn')

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

In [ ]:
summarization_model_name = 'facebook/bart-large-cnn'
summarizer_tokenizer = SummarizationTokenizer.from_pretrained(summarization_model_name)
summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(summarization_model_name).to(device)

In [ ]:
def summarize_premise(premise):
    inputs = summarizer_tokenizer(premise, return_tensors='pt', truncation=True, max_length=512).to(device)
    summary_ids = summarizer_model.generate(inputs['input_ids'], max_length=50, min_length=25, length_penalty=2.0,
                                            num_beams=4, early_stopping=True)
    summary = summarizer_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [ ]:
# def prepare_dataset_nli(examples, tokenizer, max_length):
#     examples['premise'] = [summarize_premise(premise) for premise in examples['premise']]  # Summarize on CPU
#     return tokenizer(examples['premise'], examples['hypothesis'], truncation=True,
#                      padding='max_length', max_length=max_length)

def prepare_dataset_nli(examples, tokenizer, max_length):
    examples['premise'] = [summarize_premise(premise) for premise in examples['premise']]  # Summarize premises
    tokenized_inputs = tokenizer(examples['premise'], examples['hypothesis'], truncation=True,
                                 padding='max_length', max_length=max_length)
    return {key: torch.tensor(val).to(device) for key, val in tokenized_inputs.items()}

In [ ]:
task = 'nli'  # or 'qa' for question-answering task
model_name = 'google/electra-small-discriminator'
dataset_name = 'snli'  # or 'squad' for QA task
max_length = 128
output_dir = './output'
do_train = True
do_eval = True
num_train_epochs = 3
per_device_train_batch_size = 16

In [ ]:
if task == 'nli':
    dataset_id = ('snli',)
elif task == 'qa':
    dataset_id = ('squad',)

dataset = datasets.load_dataset(*dataset_id)

In [ ]:
task_kwargs = {'num_labels': 3} if task == 'nli' else {}
model_classes = {'qa': AutoModelForQuestionAnswering, 'nli': AutoModelForSequenceClassification}
model_class = model_classes[task]
model = model_class.from_pretrained(model_name).to(device) 
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

In [ ]:
NUM_PREPROCESSING_WORKERS = 2

if do_train:
    train_dataset = dataset['train']
    train_dataset_featurized = train_dataset.map(
        lambda exs: prepare_dataset_nli(exs, tokenizer, max_length),
        batched=True,
        num_proc=1,
        remove_columns=train_dataset.column_names)

if do_eval:
    eval_split = 'validation_matched' if dataset_id == ('glue', 'mnli') else 'validation'
    eval_dataset = dataset[eval_split]
    eval_dataset_featurized = eval_dataset.map(
        lambda exs: prepare_dataset_nli(exs, tokenizer, max_length),
        batched=True,
        num_proc=1,
        remove_columns=eval_dataset.column_names)

In [ ]:
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    num_train_epochs=num_train_epochs,
    evaluation_strategy="epoch" if do_eval else "no",
)

trainer_class = Trainer

compute_metrics_fn = None

if task == 'qa':
    # For QA-specific metrics (e.g., SQuAD), use custom trainer class and metrics.
    trainer_class = QuestionAnsweringTrainer
    metric = evaluate.load('squad')
    
    def compute_metrics(eval_preds):
        return metric.compute(predictions=eval_preds.predictions, references=eval_preds.label_ids)
    
elif task == 'nli':
    def compute_metrics(eval_preds):
        predictions = eval_preds.predictions.argmax(-1)
        return {"accuracy": (predictions == eval_preds.label_ids).astype(float).mean().item()}

In [ ]:
trainer = trainer_class(
    model=model,
    args=training_args,
    train_dataset=train_dataset_featurized if do_train else None,
    eval_dataset=eval_dataset_featurized if do_eval else None,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics if do_eval else None,
)

In [ ]:
if do_train:
    trainer.train()
    trainer.save_model()

if do_eval:
    results = trainer.evaluate()
    print("Evaluation results:", results)

# Save evaluation results to file (optional)
os.makedirs(output_dir, exist_ok=True)
with open(os.path.join(output_dir, "eval_metrics.json"), "w") as f:
    json.dump(results, f)